# JSONMusicMap — Full Analysis (Colab)

**Run order:** Cell 1 → restart runtime → Cell 2 → Cell 3 → Cell 4 → … → Cell 10

Output: `music.map.json` with beats, downbeats, bars, transients, neural section labels.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# Run ONCE. Restart runtime when prompted, then continue from Cell 2.
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args])
    if r.returncode != 0:
        raise RuntimeError(f'pip {args[0]} failed — see output above')

# 1. Cython — required for madmom source build
pip('install', 'cython', '-q')

# 2. madmom
pip('install', 'madmom', '--no-build-isolation', '-q')

# 3. natten — must match Colab's PyTorch + CUDA version exactly
#    The wheel URL format: shi-labs.com/natten/wheels/cu{cuda}/torch{ver}/index.html
import torch
_tv = torch.__version__.split('+')[0].split('.')          # e.g. ['2','5','1']
_torch_tag = _tv[0] + _tv[1] + (_tv[2] if len(_tv) > 2 else '0')  # e.g. '251'
_cuda_tag  = (torch.version.cuda or 'cpu').replace('.', '')         # e.g. '121'

if _cuda_tag != 'cpu':
    _natten_url = f'https://shi-labs.com/natten/wheels/cu{_cuda_tag}/torch{_torch_tag}/index.html'
    print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
    print(f'natten wheel index: {_natten_url}')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install',
                        'natten==0.17.5', '-f', _natten_url, '-q'])
    if r.returncode != 0:
        print('⚠️  No pre-built wheel found — building natten from source (takes ~5 min)…')
        pip('install', 'natten==0.17.5', '-q')
else:
    print('No GPU detected — installing natten CPU wheel')
    pip('install', 'natten==0.17.5', '-q')

# 4. all-in-one-fix (neural section labeler + Demucs stem separator)
pip('install', 'all-in-one-fix', '--no-build-isolation', '-q')

# 5. librosa + scipy (already on Colab, but ensure fresh versions)
pip('install', 'librosa>=0.10', 'scipy', '-q')

print('\n✅ Done. Restart runtime now, then run Cell 2 onward.')

In [ ]:
# ── Cell 2: Upload your MP3 ───────────────────────────────────────────────
from google.colab import files
import os

uploaded = files.upload()   # pick your MP3
AUDIO_PATH = list(uploaded.keys())[0]
print(f'Loaded: {AUDIO_PATH}  ({os.path.getsize(AUDIO_PATH)/1e6:.1f} MB)')

In [ ]:
# ── Cell 3: Compat patches (run before any madmom/allin1 imports) ──────────
# madmom was written for numpy < 1.24 and Python < 3.10.
# These patches add back the removed aliases so madmom doesn't crash on import.

import numpy as np
import collections, collections.abc

# numpy aliases removed in 1.24
for _alias, _real in [('float',   np.float64),
                       ('complex', np.complex128),
                       ('int',     np.int64),
                       ('bool',    np.bool_),
                       ('object',  np.object_),
                       ('str',     np.str_)]:
    setattr(np, _alias, _real)

# collections aliases moved to collections.abc in Python 3.10
for _name in ('MutableSequence', 'MutableMapping', 'Mapping',
              'Callable', 'Iterator', 'Iterable', 'Sequence'):
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

print(f'numpy {np.__version__} — compat patches applied')

In [ ]:
# ── Cell 4: allin1 — neural section labels + beats ────────────────────────
# Demucs separates the track into drums/bass/vocals/other stems first.
# First run downloads ~160 MB of model weights (cached after).
# If this cell errors with "natten" in the traceback, re-run Cell 1 with GPU runtime.

import allin1fix

print(f'Running allin1 on {AUDIO_PATH} …')
print('  Step 1: Demucs stem separation (drums / bass / vocals / other)')
print('  Step 2: DiNAT neural structure analysis per stem')
a1_result = allin1fix.analyze(
    AUDIO_PATH,
    skip_separation=False,
    keep_byproducts=True,
    demix_dir='./demix',
    include_activations=False,
    include_embeddings=False,
)

print(f'\nallin1 output:')
print(f'  BPM:       {a1_result.bpm:.1f}')
print(f'  Beats:     {len(a1_result.beats)}')
print(f'  Downbeats: {len(a1_result.downbeats)}')
print(f'  Segments:')
for s in a1_result.segments:
    print(f'    [{s.label:10}] {s.start:.2f}s → {s.end:.2f}s  ({s.end-s.start:.1f}s)')

import glob
stem_files = glob.glob('./demix/**/*.wav', recursive=True)
print(f'\nSaved stems ({len(stem_files)}):')
for f in stem_files:
    print(f'  {f}')

In [ ]:
# ── Cell 5: madmom — high-accuracy RNN beat grid ──────────────────────────
# Cross-validates allin1 beat times with madmom's RNN tracker.
# madmom generally has tighter timing accuracy than allin1.

from madmom.features.beats import RNNBeatProcessor, BeatTrackingProcessor

print('Running madmom beat tracker...')
beat_act   = RNNBeatProcessor()(AUDIO_PATH)
madmom_beats = [round(float(t), 3) for t in BeatTrackingProcessor(fps=100)(beat_act)]

intervals = np.diff(madmom_beats)
madmom_bpm = round(60.0 / float(np.median(intervals)), 1)
madmom_beat_interval = round(60.0 / madmom_bpm, 4)

# Downbeats = every 4th beat from index 0
# Cross-check with allin1 downbeats to pick the phase-correct offset
a1_downbeats = [round(float(d), 3) for d in a1_result.downbeats]

# Find which offset (0,1,2,3) aligns madmom beats to allin1 downbeats
best_offset, best_score = 0, float('inf')
for offset in range(4):
    candidate_db = madmom_beats[offset::4]
    if not candidate_db or not a1_downbeats:
        continue
    # Mean nearest distance
    score = np.mean([min(abs(d - c) for c in candidate_db) for d in a1_downbeats[:20]])
    if score < best_score:
        best_score, best_offset = score, offset

madmom_downbeats = madmom_beats[best_offset::4]

print(f'  madmom BPM: {madmom_bpm}  beats: {len(madmom_beats)}  downbeats: {len(madmom_downbeats)}')
print(f'  allin1 BPM: {a1_result.bpm:.1f}  beats: {len(a1_result.beats)}  downbeats: {len(a1_downbeats)}')
print(f'  Downbeat phase offset used: {best_offset} (alignment error: {best_score*1000:.1f}ms)')

In [ ]:
# ── Cell 6: librosa — key, transients, per-section features ──────────────
# Transients are detected on the separated drums stem (much cleaner than
# filtering the full mix — no bleed from bass or vocals).
import librosa
from scipy.signal import butter, sosfilt
import glob

print('Loading audio...')
y, sr = librosa.load(AUDIO_PATH)
hop   = 512
duration = round(float(len(y) / sr), 2)

# Key from full mix
chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop)
KEYS   = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
key    = KEYS[chroma.mean(axis=1).argmax()]
print(f'  Key: {key}')

# Load separated stems from demix dir
def load_stem(pattern):
    matches = glob.glob(pattern, recursive=True)
    if not matches:
        return None, None
    y_stem, sr_stem = librosa.load(matches[0], mono=True)
    print(f'  Loaded stem: {matches[0]}')
    return y_stem, sr_stem

y_drums, sr_d = load_stem('./demix/**/drums.wav')
y_bass,  sr_b = load_stem('./demix/**/bass.wav')

# Bandpass helper (fallback for when stem not available)
def bandpass(sig, sample_rate, low_hz, high_hz=None):
    nyq = sample_rate / 2
    if high_hz is None or high_hz >= nyq:
        sos = butter(4, low_hz / nyq, btype='high', output='sos')
    else:
        sos = butter(4, [low_hz / nyq, high_hz / nyq], btype='band', output='sos')
    return sosfilt(sos, sig)

def detect_onsets(sig, sample_rate, hop_l, delta=0.07, wait=4):
    frames = librosa.onset.onset_detect(
        y=sig, sr=sample_rate, hop_length=hop_l, units='frames',
        pre_max=3, post_max=3, pre_avg=5, post_avg=5,
        delta=delta, wait=wait
    )
    return [round(float(t), 3) for t in librosa.frames_to_time(frames, sr=sample_rate, hop_length=hop_l)]

print('  Detecting transients...')
if y_drums is not None:
    # Use isolated drum stem — much cleaner separation of kick vs snare
    kick_times  = detect_onsets(bandpass(y_drums, sr_d, 40,  200),  sr_d, hop, delta=0.06, wait=6)
    snare_times = detect_onsets(bandpass(y_drums, sr_d, 200, 2000), sr_d, hop, delta=0.07, wait=6)
    hihat_times = detect_onsets(bandpass(y_drums, sr_d, 5000),      sr_d, hop, delta=0.04, wait=3)
    print('  (using separated drums stem)')
else:
    # Fallback: filter full mix
    kick_times  = detect_onsets(bandpass(y, sr, 40,  200),  sr, hop, delta=0.08, wait=6)
    snare_times = detect_onsets(bandpass(y, sr, 200, 2000), sr, hop, delta=0.09, wait=6)
    hihat_times = detect_onsets(bandpass(y, sr, 5000),      sr, hop, delta=0.05, wait=4)
    print('  (fallback: filtering full mix — drums stem not found)')

print(f'  Kick: {len(kick_times)}  Snare: {len(snare_times)}  Hi-hat: {len(hihat_times)}')

# Per-section spectral features from full mix
rms_series = librosa.feature.rms(y=y, hop_length=hop)[0]
contrast   = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=hop).mean(axis=0)
brightness = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]

# Bass energy per section (from bass stem if available)
if y_bass is not None:
    bass_rms = librosa.feature.rms(y=y_bass, hop_length=hop)[0]
else:
    bass_rms = librosa.feature.rms(y=bandpass(y, sr, 40, 250), hop_length=hop)[0]

def section_features(start, end):
    s = int(start * sr / hop)
    e = int(end   * sr / hop)
    if e <= s:
        return {}
    return {
        'rms':        round(float(np.mean(rms_series[s:e])), 5),
        'contrast':   round(float(np.mean(contrast[s:e])), 3),
        'brightness': round(float(np.mean(brightness[s:e])), 1),
        'bass_rms':   round(float(np.mean(bass_rms[s:e])), 5),
    }

print('  Spectral features computed.')

In [ ]:
# ── Cell 7: Build bars from madmom beats ─────────────────────────────────
beat_times = madmom_beats
downbeats  = madmom_downbeats
beat_interval = madmom_beat_interval

# Group beats into 4-beat bars
bars = []
bar_id = 1
for i in range(0, len(beat_times) - 3, 4):
    bars.append({
        'id':    bar_id,
        'start': beat_times[i],
        'beats': beat_times[i:i+4]
    })
    bar_id += 1

print(f'Bars: {len(bars)}')

In [ ]:
# ── Cell 8: Build sections from allin1 + snap to downbeat grid ───────────
def nearest_downbeat(t, downbeats):
    arr = np.array(downbeats)
    return float(arr[np.argmin(np.abs(arr - t))])

sections = []
for i, seg in enumerate(a1_result.segments):
    raw_start = float(seg.start)
    raw_end   = float(seg.end)

    # Snap boundaries to nearest downbeat
    start = nearest_downbeat(raw_start, downbeats) if raw_start > 0.5 else raw_start
    end   = nearest_downbeat(raw_end, downbeats)   if raw_end < duration - 0.5 else raw_end

    # Phrase boundaries = all downbeats within section
    phrase_boundaries = [d for d in downbeats if start <= d <= end]

    # Section-level kick/snare count (rhythmic density indicators)
    kicks_in  = len([t for t in kick_times  if start <= t <= end])
    snares_in = len([t for t in snare_times if start <= t <= end])
    sec_dur   = end - start

    feat = section_features(start, end)

    sections.append({
        'id':               i + 1,
        'label':            seg.label,      # allin1 neural label
        'start':            round(start, 3),
        'end':              round(end, 3),
        'duration':         round(end - start, 3),
        'duration_bars':    len([d for d in downbeats if start <= d < end]),
        'raw_start':        round(raw_start, 3),
        'raw_end':          round(raw_end, 3),
        'phrase_boundaries': [round(p, 3) for p in phrase_boundaries],
        'kick_density':     round(kicks_in  / sec_dur, 2) if sec_dur > 0 else 0,
        'snare_density':    round(snares_in / sec_dur, 2) if sec_dur > 0 else 0,
        **feat
    })

print('Sections:')
for s in sections:
    print(f"  §{s['id']} [{s['label']:10}] {s['start']}s → {s['end']}s "
          f"({s['duration_bars']} bars) "
          f"rms={s.get('rms',0):.4f} brightness={s.get('brightness',0):.0f}Hz")

In [ ]:
# ── Cell 9: Assemble + save music.map.json ────────────────────────────────
import json

music_map = {
    'file':          AUDIO_PATH,
    'duration':      duration,
    'bpm':           madmom_bpm,
    'beat_interval': beat_interval,
    'key':           key,
    'beats':         beat_times,
    'downbeats':     [round(d, 3) for d in downbeats],
    'bars':          bars,
    'transients': {
        'kick':  kick_times,
        'snare': snare_times,
        'hihat': hihat_times,
    },
    'sections': sections,
    'analysis_sources': {
        'sections':   'allin1fix (DiNAT neural net, Demucs stem separation)',
        'beats':      'madmom RNNBeatProcessor (phase-aligned to allin1 downbeats)',
        'key':        'librosa chroma_cqt (full mix)',
        'transients': 'librosa onset_detect on separated drums stem (Demucs)',
        'bass_rms':   'librosa rms on separated bass stem (Demucs)',
    }
}

OUT = 'music.map.json'
with open(OUT, 'w') as f:
    json.dump(music_map, f, indent=2)

print(f'✅ Saved {OUT}')
print(f'   BPM {music_map["bpm"]} | Key {music_map["key"]} | '
      f'{len(music_map["beats"])} beats | {len(music_map["downbeats"])} downbeats | '
      f'{len(music_map["bars"])} bars')
print(f'   Kick {len(kick_times)} | Snare {len(snare_times)} | Hi-hat {len(hihat_times)}')
print(f'   Sections:')
for s in sections:
    print(f"     §{s['id']} [{s['label']:10}] {s['start']}s → {s['end']}s "
          f"({s['duration_bars']} bars)  rms={s.get('rms',0):.4f}  bass={s.get('bass_rms',0):.4f}")

In [ ]:
# ── Cell 10: Download music.map.json ─────────────────────────────────────
from google.colab import files
files.download('music.map.json')
print('⬇️  Download started')